# Retrieval pipeline exploration

Interactive exploration of the Meridian hybrid retrieval pipeline.
Use this notebook to:
- Test retrieval quality on specific queries
- Compare dense vs BM25 vs fused results
- Visualize chunk scores and content
- Identify retrieval failures for debugging

In [ ]:
from src.config import settings
import asyncio
import sys

sys.path.insert(0, "..")

# Notebook-safe asyncio
import nest_asyncio

nest_asyncio.apply()


print(f"Meridian v{settings.VERSION} — {settings.ENVIRONMENT}")

In [ ]:
# Run a retrieval query
from src.db.session import get_db_session
from src.rag.retrieve import hybrid_retrieve

QUERY = "data retention period for personal data"
SCOPE = ["gdpr"]


async def run_retrieval():
    async with get_db_session() as session:
        return await hybrid_retrieve(session, QUERY, SCOPE)


chunks = asyncio.run(run_retrieval())
print(f"Retrieved {len(chunks)} chunks")

In [ ]:
import pandas as pd

df = pd.DataFrame(
    [
        {
            "rank": c.final_rank,
            "regulation": c.regulation,
            "article": c.article,
            "dense_score": c.dense_score,
            "bm25_score": c.bm25_score,
            "rrf_score": c.rrf_score,
            "rerank_score": c.rerank_score,
            "content_preview": c.content[:80],
        }
        for c in chunks
    ]
)
df

In [ ]:
# Visualize score distributions
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(df["rank"], df["dense_score"], color="steelblue")
axes[0].set_title("Dense retrieval scores")
axes[0].set_xlabel("Final rank")

axes[1].bar(df["rank"], df["rrf_score"], color="coral")
axes[1].set_title("RRF scores")
axes[1].set_xlabel("Final rank")

axes[2].bar(df["rank"], df["rerank_score"], color="green")
axes[2].set_title("Rerank scores (cross-encoder)")
axes[2].set_xlabel("Final rank")

plt.tight_layout()
plt.show()

In [ ]:
# Print full content of top chunks
for i, chunk in enumerate(chunks[:3], 1):
    print(f"--- Rank {i}: {chunk.regulation.upper()} {chunk.article} ---")
    print(chunk.content)
    print(f"  Rerank score: {chunk.rerank_score:.4f}\n")

## RAGAS evaluation on a small sample

In [ ]:
import json

# Load 5 golden examples
examples = []
with open("../data/golden/gdpr_qa.jsonl") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        examples.append(json.loads(line))

print(f"Loaded {len(examples)} golden examples")
print("First question:", examples[0]["question"])